# Hugging Face Loading Smoke Test

This notebook tests Notebook Lens as the only notebook mutation/execution surface for a small real Hugging Face data-loading task.

Evidence target:

- Configure Hugging Face cache paths under the round-specific scratch directory.
- Load `fancyzhx/ag_news` through a streaming path adapted from `samples/hf_streaming_text_loader.py`.
- Load a tiny materialized split to exercise first-run download/cache/progress output.
- Inspect the resulting notebook with `state`, `show-cell`, and a fresh `run-clean`.



In [1]:
from __future__ import annotations

import os
from pathlib import Path


SCRATCH = Path("tmp/agent_rounds/20260615_round6/hf_loading").resolve()
SCRATCH.mkdir(parents=True, exist_ok=True)

cache_paths = {
    "HF_HOME": SCRATCH / "hf_home",
    "HF_DATASETS_CACHE": SCRATCH / "hf_datasets",
    "HUGGINGFACE_HUB_CACHE": SCRATCH / "hf_home" / "hub",
    "TRANSFORMERS_CACHE": SCRATCH / "transformers",
}
for name, path in cache_paths.items():
    path.mkdir(parents=True, exist_ok=True)
    os.environ[name] = str(path)

env_path = Path("/Users/guraltoo/Documents/dev/proj/experiments/research-suite/.env")
token_source = "not present"
if env_path.exists():
    for raw_line in env_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, raw_value = line.split("=", 1)
        key = key.strip()
        value = raw_value.strip().strip("\"'")
        if key in {"HF_TOKEN", "HUGGINGFACE_HUB_TOKEN"} and value:
            os.environ["HF_TOKEN"] = value
            os.environ["HUGGINGFACE_HUB_TOKEN"] = value
            token_source = ".env"
            break

print(f"scratch_dir: {SCRATCH}")
for name in cache_paths:
    print(f"{name}: {os.environ[name]}")
print(f"hf_token_configured: {'yes' if os.environ.get('HF_TOKEN') else 'no'}")
print(f"hf_token_source: {token_source}")



scratch_dir: /Users/guraltoo/Documents/dev/proj/experiments/notebook-lens/tmp/agent_rounds/20260615_round6/hf_loading
HF_HOME: /Users/guraltoo/Documents/dev/proj/experiments/notebook-lens/tmp/agent_rounds/20260615_round6/hf_loading/hf_home
HF_DATASETS_CACHE: /Users/guraltoo/Documents/dev/proj/experiments/notebook-lens/tmp/agent_rounds/20260615_round6/hf_loading/hf_datasets
HUGGINGFACE_HUB_CACHE: /Users/guraltoo/Documents/dev/proj/experiments/notebook-lens/tmp/agent_rounds/20260615_round6/hf_loading/hf_home/hub
TRANSFORMERS_CACHE: /Users/guraltoo/Documents/dev/proj/experiments/notebook-lens/tmp/agent_rounds/20260615_round6/hf_loading/transformers
hf_token_configured: no
hf_token_source: not present


In [2]:
from __future__ import annotations

import html
import re
import warnings
from collections import Counter
from itertools import islice
from statistics import mean

from IPython.display import HTML, display
from datasets import load_dataset


warnings.filterwarnings("ignore", message="IProgress not found.*")

DATASET_NAME = "fancyzhx/ag_news"
DATASET_CONFIG = None
DATASET_SPLIT = "train"
TEXT_FIELD = "text"
ROW_LIMIT = 40

stream = load_dataset(DATASET_NAME, DATASET_CONFIG, split=DATASET_SPLIT, streaming=True)

lengths: list[int] = []
label_counts: Counter[int] = Counter()
token_counts: Counter[str] = Counter()
examples: list[tuple[int, int, str]] = []

for row_number, row in enumerate(islice(stream, ROW_LIMIT), start=1):
    text = str(row.get(TEXT_FIELD) or "")
    label = int(row.get("label", -1))
    lengths.append(len(text))
    label_counts[label] += 1
    token_counts.update(re.findall(r"[A-Za-z][A-Za-z0-9_'-]{2,}", text.lower())[:80])

    if len(examples) < 6:
        preview = re.sub(r"\s+", " ", text).strip()[:220]
        examples.append((row_number, label, preview))

print(f"dataset: {DATASET_NAME}")
print(f"mode: streaming")
print(f"split: {DATASET_SPLIT}")
print(f"rows_streamed: {len(lengths)}")
print(f"mean_chars: {mean(lengths):.1f}")
print(f"max_chars: {max(lengths)}")
print("label_counts:")
for label, count in sorted(label_counts.items()):
    print(f"- {label}: {count}")
print("top_tokens:")
for token, count in token_counts.most_common(12):
    print(f"- {token}: {count}")

rows = "\n".join(
    "<tr>"
    f"<td>{row_number}</td>"
    f"<td>{label}</td>"
    f"<td>{html.escape(preview)}</td>"
    "</tr>"
    for row_number, label, preview in examples
)
display(
    HTML(
        "<h3>AG News Streaming Preview</h3>"
        "<table>"
        "<tr><th>row</th><th>label</th><th>preview</th></tr>"
        f"{rows}"
        "</table>"
    )
)

HF_STREAMING_RESULT = {
    "dataset": DATASET_NAME,
    "mode": "streaming",
    "rows": len(lengths),
    "label_counts": dict(sorted(label_counts.items())),
    "mean_chars": mean(lengths),
}



/Users/guraltoo/Documents/dev/proj/experiments/notebook-lens/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


dataset: fancyzhx/ag_news
mode: streaming
split: train
rows_streamed: 40
mean_chars: 208.3
max_chars: 416
label_counts:
- 2: 40
top_tokens:
- the: 71
- and: 20
- for: 17
- reuters: 16
- oil: 14
- prices: 10
- from: 10
- market: 9
- economy: 9
- new: 9
- after: 8
- but: 8


row,label,preview
1,2,"Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again."
2,2,"Carlyle Looks Toward Commercial Aerospace (Reuters) Reuters - Private investment firm Carlyle Group,\which has a reputation for making well-timed and occasionally\controversial plays in the defense industry, has quietly"
3,2,Oil and Economy Cloud Stocks' Outlook (Reuters) Reuters - Soaring crude prices plus worries\about the economy and the outlook for earnings are expected to\hang over the stock market next week during the depth of the\summ
4,2,Iraq Halts Oil Exports from Main Southern Pipeline (Reuters) Reuters - Authorities have halted oil export\flows from the main pipeline in southern Iraq after\intelligence showed a rebel militia could strike\infrastructur
5,2,"Oil prices soar to all-time record, posing new menace to US economy (AFP) AFP - Tearaway world oil prices, toppling records and straining wallets, present a new economic menace barely three months before the US president"
6,2,"Stocks End Up, But Near Year Lows (Reuters) Reuters - Stocks ended slightly higher on Friday\but stayed near lows for the year as oil prices surged past #36;46\a barrel, offsetting a positive outlook from computer maker\"


In [3]:
from __future__ import annotations

import html
import os
from collections import Counter

from IPython.display import HTML, display
from datasets import load_dataset


small = load_dataset(
    "fancyzhx/ag_news",
    split="train[:32]",
    cache_dir=os.environ["HF_DATASETS_CACHE"],
)

labels = Counter(int(item["label"]) for item in small)
cache_files = small.cache_files
print(f"dataset_type: {type(small).__name__}")
print(f"rows_loaded: {small.num_rows}")
print(f"features: {small.features}")
print(f"fingerprint: {small._fingerprint}")
print(f"cache_files: {len(cache_files)}")
for entry in cache_files[:3]:
    print(f"- cache_file: {entry.get('filename')}")
print("materialized_label_counts:")
for label, count in sorted(labels.items()):
    print(f"- {label}: {count}")

rows = "\n".join(
    "<tr>"
    f"<td>{idx}</td>"
    f"<td>{item['label']}</td>"
    f"<td>{html.escape(item['text'][:180])}</td>"
    "</tr>"
    for idx, item in enumerate(small.select(range(min(5, small.num_rows))), start=1)
)
display(
    HTML(
        "<h3>Materialized AG News Sample</h3>"
        "<table>"
        "<tr><th>row</th><th>label</th><th>text</th></tr>"
        f"{rows}"
        "</table>"
    )
)

HF_MATERIALIZED_RESULT = {
    "rows": small.num_rows,
    "cache_files": [entry.get("filename") for entry in cache_files],
    "labels": dict(sorted(labels.items())),
}



dataset_type: Dataset
rows_loaded: 32
features: {'text': Value('string'), 'label': ClassLabel(names=['World', 'Sports', 'Business', 'Sci/Tech'])}
fingerprint: ddd95ca9bf910afa
cache_files: 1
- cache_file: /Users/guraltoo/Documents/dev/proj/experiments/notebook-lens/tmp/agent_rounds/20260615_round6/hf_loading/hf_datasets/fancyzhx___ag_news/default/0.0.0/eb185aade064a813bc0b7f42de02595523103ca4/ag_news-train.arrow
materialized_label_counts:
- 2: 32


row,label,text
1,2,"Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again."
2,2,"Carlyle Looks Toward Commercial Aerospace (Reuters) Reuters - Private investment firm Carlyle Group,\which has a reputation for making well-timed and occasionally\controversial pla"
3,2,Oil and Economy Cloud Stocks' Outlook (Reuters) Reuters - Soaring crude prices plus worries\about the economy and the outlook for earnings are expected to\hang over the stock marke
4,2,Iraq Halts Oil Exports from Main Southern Pipeline (Reuters) Reuters - Authorities have halted oil export\flows from the main pipeline in southern Iraq after\intelligence showed a
5,2,"Oil prices soar to all-time record, posing new menace to US economy (AFP) AFP - Tearaway world oil prices, toppling records and straining wallets, present a new economic menace bar"


In [4]:
from __future__ import annotations

import os
from pathlib import Path


SCRATCH = Path("tmp/agent_rounds/20260615_round6/hf_loading").resolve()


def inventory(path: Path) -> tuple[int, int]:
    files = [item for item in path.rglob("*") if item.is_file()]
    return len(files), sum(item.stat().st_size for item in files)


print(f"scratch_dir: {SCRATCH}")
for name in ["HF_HOME", "HF_DATASETS_CACHE", "HUGGINGFACE_HUB_CACHE", "TRANSFORMERS_CACHE"]:
    path = Path(os.environ[name])
    file_count, byte_count = inventory(path)
    inside_scratch = path.resolve().is_relative_to(SCRATCH)
    print(
        f"{name}: exists={path.exists()} inside_scratch={inside_scratch} "
        f"files={file_count} bytes={byte_count}"
    )

print(f"streaming_rows: {HF_STREAMING_RESULT['rows']}")
print(f"materialized_rows: {HF_MATERIALIZED_RESULT['rows']}")



scratch_dir: /Users/guraltoo/Documents/dev/proj/experiments/notebook-lens/tmp/agent_rounds/20260615_round6/hf_loading
HF_HOME: exists=True inside_scratch=True files=13 bytes=39662170
HF_DATASETS_CACHE: exists=True inside_scratch=True files=3 bytes=31726355
HUGGINGFACE_HUB_CACHE: exists=True inside_scratch=True files=11 bytes=39656905
TRANSFORMERS_CACHE: exists=True inside_scratch=True files=0 bytes=0
streaming_rows: 40
materialized_rows: 32


# Evidence Summary

Notebook Lens successfully created and executed a readable notebook for a bounded Hugging Face data-loading task.

Observed evidence:

- The setup cell placed `HF_HOME`, `HF_DATASETS_CACHE`, `HUGGINGFACE_HUB_CACHE`, and `TRANSFORMERS_CACHE` under the requested scratch directory.
- The streaming cell loaded 40 rows from `fancyzhx/ag_news`, printed compact aggregate statistics, and rendered an HTML preview table.
- The materialized cell loaded `train[:32]`, reported features, label counts, fingerprint, and cache file paths.
- The cache inventory cell confirmed all configured Hugging Face cache paths stayed under the scratch directory.

The notebook is intentionally short: one setup cell, two loading/evidence cells, one cache inventory cell, and this narrative summary.



# CLI Inspection Notes

Notebook Lens inspection commands were useful after execution:

- `state --outputs summary` listed markdown/code cells with stable cell ids, compact source excerpts, stdout, stderr, and HTML previews.
- `show-cell --id 7cf534ae --outputs full` preserved the streaming cell evidence, including separated `stderr` for the `IProgress` and unauthenticated Hugging Face warnings.
- `show-cell --id 3133f62d --outputs full` preserved the first materialized-load progress bars before `run-clean`; after `run-clean`, the cached rerun was quieter.
- `run-clean --timeout 300` completed successfully from a fresh kernel, executing all 4 code cells.
- No Hugging Face token value was printed. The setup cell reported only whether a token was configured.

